# Pixora v1: экономика пакета доступа Pixora

Воспроизводимая модель цифрового продукта «Пакет доступа Pixora»: новый пользователь получает две успешные обработки, каждый пакет за 49 ₽ включает ещё две обработки и одно право на выбранный оригинал. Неиспользованные внутренние credits не считаются расходом. Все финансовые ставки ниже — рабочие допущения, а не данные провайдера или Robokassa.

In [1]:
from dataclasses import dataclass, asdict

PRICE_RUB = 49.0
PAYMENT_FEE_RATE = 0.05
TAX_RATE = 0.04
SUPPORT_AND_REFUND_RESERVE_RUB = 5.0
GENERATION_COST_RUB = 10.0

NET_PACK_BEFORE_GENERATION = (
    PRICE_RUB * (1 - PAYMENT_FEE_RATE - TAX_RATE)
    - SUPPORT_AND_REFUND_RESERVE_RUB
)
print({'net_pack_before_generation_rub': round(NET_PACK_BEFORE_GENERATION, 2)})

{'net_pack_before_generation_rub': 39.59}


In [2]:
@dataclass(frozen=True)
class Scenario:
    name: str
    free_generations_used: float
    packs_bought: float
    paid_generations_used: float
    originals_unlocked: float

SCENARIOS = [
    Scenario('1 free, then one pack', 1, 1, 0, 0),
    Scenario('2 free, no purchase', 2, 0, 0, 0),
    Scenario('2 free + one fully generated pack', 2, 1, 2, 1),
    Scenario('2 free + two fully generated packs', 2, 2, 4, 2),
    Scenario('original used, paid generations unused', 2, 1, 0, 1),
    Scenario('paid generations used, original unused', 2, 1, 2, 0),
    Scenario('purchased pack completely unused', 2, 1, 0, 0),
]

def evaluate(s):
    revenue = s.packs_bought * PRICE_RUB
    variable_generation_cost = (s.free_generations_used + s.paid_generations_used) * GENERATION_COST_RUB
    contribution = s.packs_bought * NET_PACK_BEFORE_GENERATION - variable_generation_cost
    return {**asdict(s), 'revenue_rub': round(revenue, 2), 'generation_cost_rub': round(variable_generation_cost, 2), 'contribution_rub': round(contribution, 2)}

results = [evaluate(s) for s in SCENARIOS]
for row in results:
    print(row)

{'name': '1 free, then one pack', 'free_generations_used': 1, 'packs_bought': 1, 'paid_generations_used': 0, 'originals_unlocked': 0, 'revenue_rub': 49.0, 'generation_cost_rub': 10.0, 'contribution_rub': 29.59}
{'name': '2 free, no purchase', 'free_generations_used': 2, 'packs_bought': 0, 'paid_generations_used': 0, 'originals_unlocked': 0, 'revenue_rub': 0.0, 'generation_cost_rub': 20.0, 'contribution_rub': -20.0}
{'name': '2 free + one fully generated pack', 'free_generations_used': 2, 'packs_bought': 1, 'paid_generations_used': 2, 'originals_unlocked': 1, 'revenue_rub': 49.0, 'generation_cost_rub': 40.0, 'contribution_rub': -0.41}
{'name': '2 free + two fully generated packs', 'free_generations_used': 2, 'packs_bought': 2, 'paid_generations_used': 4, 'originals_unlocked': 2, 'revenue_rub': 98.0, 'generation_cost_rub': 60.0, 'contribution_rub': 19.18}
{'name': 'original used, paid generations unused', 'free_generations_used': 2, 'packs_bought': 1, 'paid_generations_used': 0, 'origina

In [3]:
def cohort_unit_economics(*, conversion, avg_free_generations, packs_per_payer, paid_generations_per_pack):
    free_cost = avg_free_generations * GENERATION_COST_RUB
    pack_contribution = NET_PACK_BEFORE_GENERATION - paid_generations_per_pack * GENERATION_COST_RUB
    contribution_per_new_user = conversion * packs_per_payer * pack_contribution - free_cost
    break_even_conversion = free_cost / (packs_per_payer * pack_contribution) if pack_contribution > 0 else float('inf')
    return {
        'pack_contribution_rub': round(pack_contribution, 2),
        'contribution_per_new_user_rub': round(contribution_per_new_user, 2),
        'break_even_conversion_pct': round(100 * break_even_conversion, 1),
    }

for conversion in (0.10, 0.20, 0.30, 0.50):
    print(conversion, cohort_unit_economics(conversion=conversion, avg_free_generations=1.5, packs_per_payer=1.0, paid_generations_per_pack=1.5))

0.1 {'pack_contribution_rub': 24.59, 'contribution_per_new_user_rub': -12.54, 'break_even_conversion_pct': 61.0}
0.2 {'pack_contribution_rub': 24.59, 'contribution_per_new_user_rub': -10.08, 'break_even_conversion_pct': 61.0}
0.3 {'pack_contribution_rub': 24.59, 'contribution_per_new_user_rub': -7.62, 'break_even_conversion_pct': 61.0}
0.5 {'pack_contribution_rub': 24.59, 'contribution_per_new_user_rub': -2.71, 'break_even_conversion_pct': 61.0}


## Интерпретация

При рабочих допущениях один полностью использованный пакет оставляет 19,59 ₽ после комиссии, налога, резерва и двух обработок. Однако бесплатная выдача создаёт расход у всех пользователей, поэтому модель прибыльна только при достаточно высокой конверсии или меньшей фактической стоимости обработки. Оригинал не создаёт новый provider cost: он повторно доставляет уже сохранённый private original. Перед реальными платежами нужно заменить допущения фактическими invoice/usage, договорной комиссией и измеренными коэффициентами использования.